In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. Generate Synthetic Coastal Baseline Data (n=1000 households)
np.random.seed(42)
n_households = 1000

# Upazilas assigned vulnerability baselines (matching your QGIS map)
upazilas = ['Shyamnagar', 'Assasuni', 'Kaliganj', 'Debhata', 'Tala', 'Satkhira Sadar']
cvi_ranges = {
    'Shyamnagar': (0.76, 0.91),    # Very High
    'Assasuni': (0.61, 0.76),      # High
    'Kaliganj': (0.47, 0.61),      # Moderate
    'Debhata': (0.32, 0.47),       # Low
    'Tala': (0.18, 0.32),          # Very Low
    'Satkhira Sadar': (0.18, 0.32) # Very Low
}

# Assign households to upazilas (weighted toward vulnerable coastal areas)
df = pd.DataFrame({'Household_ID': range(1, n_households + 1)})
df['Upazila'] = np.random.choice(upazilas, n_households, p=[0.3, 0.25, 0.15, 0.15, 0.075, 0.075])

# Assign specific CVI score based on upazila gradient
df['CVI_Score'] = df['Upazila'].apply(lambda x: np.random.uniform(cvi_ranges[x][0], cvi_ranges[x][1]))

# 2. Simulate Structuralist Network Variables 
# Theory: Higher environmental vulnerability limits physical access to support networks
df['Ego_Network_Size'] = np.random.poisson(lam=6) - (df['CVI_Score'] * 4).astype(int)
df['Ego_Network_Size'] = df['Ego_Network_Size'].clip(lower=0)

# Simulate Environmental Shock (Higher CVI = higher probability of shock)
df['Shock_Experienced'] = np.random.binomial(1, df['CVI_Score'])

# 3. Model Migration Intent (The Dependent Variable)
# Migration intent increases with CVI & Shock, but decreases if Network Size is robust
logit_prob = -2.5 + (3.5 * df['CVI_Score']) + (1.2 * df['Shock_Experienced']) - (0.5 * df['Ego_Network_Size'])
prob_migration = 1 / (1 + np.exp(-logit_prob))
df['Migration_Intent'] = np.random.binomial(1, prob_migration)

# 4. Run the Statistical Analysis (Logistic Regression)
X = df[['CVI_Score', 'Ego_Network_Size', 'Shock_Experienced']]
X = sm.add_constant(X)
y = df['Migration_Intent']

model = sm.Logit(y, X).fit()

# Print professional output
print("\n=== Coastal Vulnerability & Network Migration Analysis ===")
print(model.summary())

# Optional: Save the synthetic dataset to CSV
df.to_csv('satkhira_synthetic_migration_data.csv', index=False)

Optimization terminated successfully.
         Current function value: 0.418395
         Iterations 7

=== Coastal Vulnerability & Network Migration Analysis ===
                           Logit Regression Results                           
Dep. Variable:       Migration_Intent   No. Observations:                 1000
Model:                          Logit   Df Residuals:                      996
Method:                           MLE   Df Model:                            3
Date:                Mon, 21 Sep 2026   Pseudo R-squ.:                  0.2097
Time:                        13:44:32   Log-Likelihood:                -418.40
converged:                       True   LL-Null:                       -529.43
Covariance Type:            nonrobust   LLR p-value:                 7.183e-48
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 1.0596      